In [ ]:
#| default_exp core

# Core
> The `FastHTML` subclass of `Starlette`.

#| export
Create an app with `app = FastHTML()`. The `@rt` decorator adds routes (`rt = app.route`), inferring what it can from the function:

    @rt
    def index(): ...      # GET,POST /
    @rt
    def foo(): ...        # GET,POST /foo
    @rt('/hi')
    def get(): ...        # GET /hi (function named after a verb handles just that verb)
    @rt('/cards/{id}')
    class Cards: ...      # class route group: each verb-named method (get, patch, ...) becomes a handler

The decorated function doubles as a URL builder, so links never need hardcoded paths:

    A('More', href=foo.to(a=1))   # <a href="/foo?a=1"> `.to()` fills path params; the rest become query params.
    Div('...', hx_get=foo)        # route functions work directly as htmx attr values: hx-get="/foo"

Handlers return FT trees (from `fastcore.xml`), rendered as HTML automatically: a full page for regular requests, a bare fragment for HTMX requests. In a full page, top-level `Title`, `Meta`, `Link`, and `Style` items go in `<head>`; the rest goes in `<body>`. Strings are sent as HTML, dicts as JSON, and a Starlette `Response` is sent untouched. A return tuple may mix content with special items: any `HttpHeader` (e.g. from `cookie()` or `HtmxResponseHeaders()`) becomes a response header, any `BackgroundTask` runs after the response is sent, and the rest renders as content. `Redirect` picks the right redirect mechanism for HTMX vs regular requests, and `FtResponse` wraps FT content when you need the status code or headers.

Handler parameters are filled from the request: each is looked up in path, cookie, header (snake_case names match Hyphen-Case headers), query, then form data, and cast by calling its annotation on the value. Hyphens in form/query names match underscores in handler parameters and typed body fields. To read a different request name, use `Annotated[T, Alias('name')]`. Exact names take precedence within each source. Dictionary and request access retain the original keys. `bool`, `int`, and `date` get smart string parsing; `UploadFile` passes through. Repeated params take the last value unless annotated `list[T]`, which collects them all. A param with no annotation and no special name is ignored, with a warning. A dataclass, TypedDict, namedtuple, or any annotated class collects the whole form body; a `__from_request__` classmethod customizes construction. A missing required param is a 400, a failed cast a 404. These special names need no annotation:

    req, ws   the Request / WebSocket connection
    sess      session dict
    app       the FastHTML app
    state     app.state
    scope     ASGI scope
    auth      scope['auth']
    htmx      HtmxHeaders (parsed HX-* request headers)
    body      raw body text
    data      parsed form data as a dict
    api       ApiReturn (API vs browser dual responses)
    send      websocket send function
    hdrs,ftrs the per-request copies of the app's `hdrs`/`ftrs` (head and footer content)
    htmlkw    the per-request copy of the app's `<html>` attrs
    bodykw    the per-request copy of the app's `<body>` attrs
    resp      the handler's response (injected into `after` functions)

`before`/`after` functions (app-level, with `Beforeware` skip patterns, or per-route `before=`) get the same param injection as handlers. A `before` returning a value responds with it and skips the handler (the usual auth-guard pattern).

See the [Handling Handlers tutorial](https://www.fastht.ml/docs/ref/handlers.html.md) for worked examples of the above.

This is the source code to fasthtml. You won't need to read this unless you want to understand how things are built behind the scenes, or need full details of a particular API. The notebook is converted to the Python module [fasthtml/core.py](https://github.com/AnswerDotAI/fasthtml/blob/main/fasthtml/core.py) using [nbdev](https://nbdev.fast.ai/).

## Imports and utils

In [ ]:
#| export
import json,uuid,inspect,types,asyncio,inspect,random,contextlib,itsdangerous,hashlib,signal,subprocess
from uuid import uuid5, NAMESPACE_URL

from fastcore.utils import *
from fastcore.xml import *
from fastcore.meta import use_kwargs_dict,delegates,splice_sig
from fastcore.style import S

from types import UnionType, SimpleNamespace as ns, GenericAlias
from typing import get_args, get_origin, Union, Mapping, List, Any, Callable, Annotated
from datetime import datetime,date
from dataclasses import dataclass
from inspect import Parameter,get_annotations
from functools import partialmethod, update_wrapper
from http import cookies
from urllib.parse import urlencode, parse_qs, quote, unquote, urlsplit, urlunsplit
from copy import deepcopy
from warnings import warn
from dateutil import parser as dtparse
from anyio import from_thread
from uuid import uuid4, UUID
from base64 import b64encode,b64decode
from email.utils import format_datetime

from fasthtml.starlette import *
from python_multipart.exceptions import MultipartParseError

In [ ]:
import time, tempfile

from IPython import display
from enum import Enum
from pprint import pprint
from collections import namedtuple
from typing import Optional, TypedDict

from starlette.testclient import TestClient
from fastcore.test import *
from fastcore.net import run_sync
from starlette.datastructures import UploadFile

In [ ]:
#| export
def _params(f): return signature_ex(f, True).parameters

empty = Parameter.empty

We write source code _first_, and then tests come _after_. The tests serve as both a means to confirm that the code works and also serves as working examples. The first exported function, `parsed_date`, is an example of this pattern.

In [ ]:
#| export
def parsed_date(s:str):
    "Convert `s` to a datetime"
    return dtparse.parse(s)

In [ ]:
parsed_date('2pm')

In [ ]:
isinstance(date.fromtimestamp(0), date)

In [ ]:
#| export
def snake2hyphens(s:str):
    "Convert `s` from snake case to hyphenated and capitalised"
    s = snake2camel(s)
    return camel2words(s, '-')

In [ ]:
snake2hyphens("snake_case")

In [ ]:
#| export
htmx_hdrs = dict(
    boosted="HX-Boosted",
    current_url="HX-Current-URL",
    history_restore_request="HX-History-Restore-Request",
    prompt="HX-Prompt",
    request="HX-Request",
    request_type="HX-Request-Type", # v4: partial or full update
    source="HX-Source", # v4: replaces v2 HX-Trigger but with tag#id instead just id
    target="HX-Target",
    trigger_name="HX-Trigger-Name",
    trigger="HX-Trigger")

@dataclass
class HtmxHeaders:
    "Parsed HX-* request headers"
    boosted:str|None=None; current_url:str|None=None; history_restore_request:str|None=None; prompt:str|None=None
    request:str|None=None; request_type:str|None=None; source:str|None=None; target:str|None=None; 
    trigger_name:str|None=None; trigger:str|None=None
    def __bool__(self): return any(hasattr(self,o) for o in htmx_hdrs)

def _get_header(h, k):
    k = k.lower()
    res = h.get(k)
    encoded = h.get(k+'-uri-autoencoded')=='true' or (k=='hx-prompt' and h.get('hx-request-type'))
    return unquote(res) if res is not None and encoded else res

def _get_htmx(h):
    res = {k:_get_header(h, v) for k,v in htmx_hdrs.items()}
    return HtmxHeaders(**res)

In [ ]:
def test_request(url: str='/', headers: dict=None, method: str='get')->Request:
    if headers is None: headers={}
    scope = dict(type='http', method=method, path=url, headers=Headers(headers).raw, query_string=b'', scheme='http',
        client=('127.0.0.1', 8000), server=('127.0.0.1', 8000))
    async def receive(): return dict(type="http.request", body=b"", more_body=False)
    return Request(scope, receive)

In [ ]:
h = test_request(headers=Headers({'HX-Request':'1'}))
_get_htmx(h.headers)

In [ ]:
#| export
def _mk_list(t, v): return [t(o) for o in listify(v)]

## Request and response

In [ ]:
#| export
fh_cfg = AttrDict(indent=True)

In [ ]:
#| export
_special_names = {'ws','request','session','scope','data','htmx','app','state','auth','send','api','body','hdrs','ftrs','bodykw','htmlkw','resp','self'}

def _check_anno(arg, anno):
    "Check for common annotation issues; returns warning string or None"
    if anno is empty and arg.lower() not in _special_names and not any(s.startswith(arg.lower()) for s in ('request','session')): return f"`{arg}` has no type annotation and is not a recognised special name, so is ignored."
    if isinstance(anno, type) and not get_origin(anno) and issubclass(anno, (list, tuple)) and not _is_body(anno): return f"`{arg}` uses bare `{anno.__name__}` annotation, so is ignored (use e.g. `{anno.__name__}[str]` instead)."

In [ ]:
#| export
def _fix_anno(t, o):
    "Create appropriate callable type for casting a `str` to type `t` (or first type in `t` if union)"
    origin = get_origin(t)
    if origin is Union or origin is UnionType: origin = get_origin(t:=first(o for o in get_args(t) if o!=type(None)))
    if origin in (list,List): t = first(o for o in get_args(t) if o!=type(None))
    d = {bool: str2bool, int: str2int, date: str2date, UploadFile: noop}
    res = d.get(t, t)
    if origin in (list,List): return _mk_list(res, o)
    if isinstance(t, type) and issubclass(t, (list,tuple)): return None
    if not isinstance(o, (str,list,tuple)): return o
    return res(o[-1]) if isinstance(o,(list,tuple)) else res(o)

In [ ]:
test_eq(_fix_anno(Union[str,None], 'a'), 'a')
test_eq(_fix_anno(float, 0.9), 0.9)
test_eq(_fix_anno(int, '1'), 1)
test_eq(_fix_anno(int, ['1','2']), 2)
test_eq(_fix_anno(list[int], ['1','2']), [1,2])
test_eq(_fix_anno(list[int], '1'), [1])
test_eq(_fix_anno(Optional[list[str]], ['1','2']), ['1','2'])
test_eq(_fix_anno(list[int] | None, ['1','2']), [1,2])
test_eq(_fix_anno(list, ['1','2']), None)
test_eq(_fix_anno(tuple, ['1','2']), None)
test_eq(_fix_anno(list[str], []), [])
test_eq(_fix_anno(Optional[list[str]], '1'), ['1'])

In [ ]:
#| export
def _form_arg(k, v, d):
    "Get type by accessing key `k` from `d`, and use to cast `v`"
    if v is None: return
    if not isinstance(v, (str,list,tuple)): return v
    # This is the type we want to cast `v` to
    anno = d.get(k, None)
    if not anno: return v
    return _fix_anno(anno, v)

In [ ]:
d = dict(k=int, l=List[int])
test_eq(_form_arg('k', "1", d), 1)
test_eq(_form_arg('l', "1", d), [1])
test_eq(_form_arg('l', ["1","2"], d), [1,2])

In [ ]:
#| export
@dataclass
class HttpHeader:
    "A response header as a k,v pair"
    k:str;v:str

In [ ]:
#| export
def _to_htmx_header(s): return 'HX-' + s.replace('_', '-').title()

htmx_resps = dict(location=None, push_url=None, redirect=None, refresh=None, replace_url=None,
    reswap=None, retarget=None, reselect=None, trigger=None, trigger_after_settle=None, trigger_after_swap=None)

In [ ]:
_to_htmx_header('trigger_after_settle')

In [ ]:
#| export
@use_kwargs_dict(**htmx_resps)
def HtmxResponseHeaders(**kwargs):
    "HTMX response headers"
    res = tuple(HttpHeader(_to_htmx_header(k), v) for k,v in kwargs.items())
    return res[0] if len(res)==1 else res

In [ ]:
HtmxResponseHeaders(trigger_after_settle='hi')

In [ ]:
#| export
def _annotations(anno):
    "Same as `get_annotations`, but also works on namedtuples"
    if is_namedtuple(anno): return {o:str for o in anno._fields}
    return get_annotations(anno)

In [ ]:
@dataclass
class Foo: a:int; b:str
_annotations(Foo)

In [ ]:
Point = namedtuple('Point', ['x', 'y'])
_annotations(Point)

In [ ]:
#| export
def _is_body(anno): return issubclass(anno, (dict,ns)) or hasattr(anno,'__from_request__') or _annotations(anno)

In [ ]:
#| export
def _formitem(form, k):
    "Return item `k`, accepting hyphens for underscores and unwrapping single-item form lists"
    if k not in form: k = {o.replace('-', '_'):o for o in form}.get(k, k)
    if isinstance(form, dict): return form.get(k)
    o = form.getlist(k)
    return o[0] if len(o) == 1 else o if o else None

In [ ]:
#| export
def form2dict(form: FormData) -> dict:
    "Convert starlette form data to a dict"
    if isinstance(form, dict): return form
    return {k: _formitem(form, k) for k in form}

In [ ]:
d = [('a',1),('a',2),('b',0)]
fd = FormData(d)
res = form2dict(fd)
test_eq(res['a'], [1,2])
test_eq(res['b'], 0)

In [ ]:
#| export
DEF_MAXPART = 100*1024*1024

In [ ]:
#| export
async def parse_form(req: Request) -> FormData:
    "Starlette errors on empty multipart/json forms and 500s on malformed multipart, so this handles those situations"
    ctype = req.headers.get("Content-Type", "")
    maxpart = getattr(req, "max_part_size", DEF_MAXPART)
    if ctype.startswith("multipart/form-data"):
        try: boundary = ctype.split("boundary=")[1].strip()
        except IndexError: raise HTTPException(400, "Invalid form-data: no boundary")
        if int(req.headers.get("Content-Length", "0")) <= len(boundary) + 6: return FormData()
        try: return await req.form(max_part_size=maxpart)
        except MultipartParseError as e: raise HTTPException(400, f"Invalid form-data: {e}")
    body = await req.body()  # Cache body for non-multipart request types
    if ctype == 'application/json':
        try: res = await req.json() if body else {}
        except ValueError: raise HTTPException(400, "Invalid JSON body")
        if not isinstance(res, dict): raise HTTPException(400, "JSON body must be an object")
        return res
    return await req.form(max_part_size=maxpart)

Use the special `__from_request__` class method for custom creation:

In [ ]:
class Foo(BasicRepr):
    def __init__(self, d): self.d = d

    @classmethod
    def __from_request__(cls, data, req): return cls(data)

In [ ]:
p = Parameter('a', Parameter.POSITIONAL_OR_KEYWORD, annotation=Foo)

req = test_request(method='POST')
async def mock_form(**kwargs): return FormData({'a':1})
req.form = mock_form

In [ ]:
#| export
async def _from_body(conn, p, data):
    "Create an instance of the annotated type from pre-parsed `data`"
    anno = p.annotation
    # param params take precedence
    data = dict(data) | getattr(conn, 'path_params', {})
    ctor = getattr(anno, '__from_request__', None)
    if ctor:
        ps = {k:v for k,v in _params(ctor).items() if k != 'cls'}
        kwargs = await _find_ps(conn, data, conn.headers, ps)
        return await maybe_await(ctor(**kwargs))
    d = _annotations(anno)
    if d: data = {k.replace('-', '_'):v for k,v in data.items()} | data
    cargs = {k: _form_arg(k, v, d) for k, v in data.items() if not d or k in d}
    return anno(**cargs)

In [ ]:
#| export
class ApiReturn:
    "Request param: call with browser and API responses; returns kwargs as JSON if the client accepts `application/json`, else `norm`"
    @classmethod
    async def __from_request__(cls, data, req): return cls(req.headers.get('accept')=='application/json')
    def __init__(self, isapi=False): self.isapi = isapi
    def __call__(self, norm=None, **kw): return kw if self.isapi else norm
    def __bool__(self): return bool(self.isapi)

Routes often need to return different responses depending on whether the request comes from a browser or an API client. The `api:ApiReturn` parameter handles this automatically — add it to any route signature and call it with two arguments: the API response and the browser response. If the request's `Accept` header is `application/json` then a dict of the kwargs is returned, otherwise, the first value is returned. For example, `return api(Redirect('/records'), id=record.id)` returns {'id':id} for API callers and a redirect for the browser. This keeps your route logic in one place instead of splitting it across separate endpoints.

In [ ]:
#| export
class JSONResponse(JSONResponseOrig):
    "Same as starlette's version, but auto-stringifies non serializable types"
    def render(self, content:Any)->bytes:
        def _default(o): return list(o) if is_listy(o) else str(o)
        res = json.dumps(content, ensure_ascii=False, allow_nan=False, indent=None, separators=(",",":"), default=_default)
        return res.encode("utf-8")


In [ ]:
async def f(req):
    def _f(p:HttpHeader): ...
    p = first(_params(_f).values())
    data = form2dict(await parse_form(req))
    result = await _from_body(req, p, data)
    return JSONResponse(result.__dict__)

client = TestClient(Starlette(routes=[Route('/', f, methods=['POST'])]))

d = dict(k='value1',v=['value2','value3'])
response = client.post('/', data=d)
print(response.json())

In [ ]:
# An empty `application/json` body yields `{}` (like the empty-multipart guard), not a 500
async def _empty_json(req): return JSONResponse(form2dict(await parse_form(req)))
client = TestClient(Starlette(routes=[Route('/', _empty_json, methods=['POST'])]))
res = client.post('/', headers={'content-type':'application/json'})
test_eq(res.status_code, 200)
test_eq(res.json(), {})
res = client.post('/', json={'a':1})          # non-empty json still parses (behavior preserved)
test_eq(res.json(), {'a':1})

A malformed multipart body (here the opening boundary isn't followed by CRLF) is the client's error, so `parse_form` answers 400 rather than letting the parser's exception surface as a 500:

In [ ]:
bad = b'--xyzXX\r\nContent-Disposition: form-data; name="a"\r\n\r\n1\r\n--xyz--\r\n'
res = client.post('/', content=bad, headers={'content-type':'multipart/form-data; boundary=xyz'})
test_eq(res.status_code, 400)
res.text

A JSON body that does not decode, or decodes to something other than an object (an array, say), cannot supply route parameters, so it is refused with 400 rather than failing inside the form conversion:

In [ ]:
res = client.post('/', json=[1,2])
test_eq(res.status_code, 400)
res = client.post('/', content=b'not json', headers={'content-type':'application/json'})
test_eq(res.status_code, 400)
res.text

In [ ]:
async def f(req): return Response(str(req.query_params.getlist('x')))
client = TestClient(Starlette(routes=[Route('/', f, methods=['GET'])]))
client.get('/?x=1&x=2').text

In [ ]:
#| export
@dataclass(frozen=True)
class Alias:
    "Read a handler param from request data under `name` instead of its Python name, e.g. `Annotated[str, Alias('user-id')]`"
    name: str

async def _find_p(conn, data, hdrs, arg:str, p:Parameter):
    "In `data` find param named `arg` of type in `p` (`arg` is ignored for body types)"
    alias = None
    if get_origin(p.annotation) is Annotated:
        anno, *meta = get_args(p.annotation)
        p = p.replace(annotation=anno)
        alias = next((m.name for m in meta if isinstance(m, Alias)), None)
    key = alias or arg
    anno = p.annotation
    # Special annotation types
    if isinstance(anno, type) and not isinstance(anno, GenericAlias):
        if issubclass(anno, HtmxHeaders): return _get_htmx(hdrs)
        if issubclass(anno, Starlette): return conn.scope['app']
        if issubclass(anno, HTTPConnection): return conn
        if issubclass(anno, State): return conn.scope['app'].state
        if anno is dict: return data
        if _is_body(anno):
            if 'session'.startswith(key.lower()): return conn.scope.get('session', {})
            return await _from_body(conn, p, data)
    if (msg := _check_anno(key, anno)): return warn(msg)
    # Special param names with no annotations
    if anno is empty:
        if key.lower()=='ws' or 'request'.startswith(key.lower()): return conn
        if 'session'.startswith(key.lower()): return conn.scope.get('session', {})
        if key.lower()=='scope': return conn.scope
        if key.lower()=='data': return data
        if key.lower()=='htmx': return _get_htmx(hdrs)
        if key.lower()=='app': return conn.scope['app']
        if key.lower()=='state': return conn.scope['app'].state
        if key.lower()=='auth': return conn.scope.get('auth', None)
        if key.lower()=='send':
            assert not isinstance(conn, Request), "`send` requires a websocket, not a `Request`"
            return partial(_send_ws, conn)
        if key.lower()=='api': return ApiReturn(hdrs.get('accept')=='application/json')
        if key.lower()=='body': return (await conn.body()).decode()
        if key.lower() in ('hdrs','ftrs','bodykw','htmlkw'): return getattr(conn, key.lower())
        return None
    # Not a special name or a special annotation
    res = conn.path_params.get(key, None)
    if res in (empty,None): res = conn.cookies.get(key, None)
    if res in (empty,None): res = _get_header(hdrs, snake2hyphens(key))
    if res in (empty,None): res = _formitem(conn.query_params, key)
    if res in (empty,None): res = _formitem(data, key)
    if res in (empty,None) and isinstance(vals := data.get('values'), dict): res = _formitem(vals, key)  # htmx v4 nests form data under `values`
    if res in (empty,None):
        if p.default is empty:
            if isinstance(conn, Request): raise HTTPException(400, f"Missing required field: {key}")
            raise ValueError(f"Missing required field: {key}")
        res = p.default
    try: return _fix_anno(anno, res)
    except ValueError as e:
        if isinstance(conn, Request): raise HTTPException(404, f"{conn.url.path}: {e}") from None
        raise

In [ ]:
#| export
async def _find_ps(conn, data, hdrs, params):
    if conn.query_params: data |= dict(conn.query_params)
    return {arg:await _find_p(conn, data, hdrs, arg, p) for arg,p in params.items()}

In [ ]:
data = form2dict(await parse_form(req))
await _from_body(req, p, data)

In [ ]:
#| export
async def _wrap_req(req, params):
    data = form2dict(await parse_form(req))
    return await _find_ps(req, data, req.headers, params)

In [ ]:
def g(test_val:int=0, item:HttpHeader=None, htmx:HtmxHeaders=None, api:ApiReturn=None): ...

async def f(req):
    await _wrap_req(req, _params(g))
    body = await req.body()
    return Response(f"ok body={len(body)}")

client = TestClient(Starlette(routes=[Route('/', f, methods=['POST'])]))
r = client.post('/', data=dict(test_val='21', k='a', v='b'), headers={'HX-Request': '1'})
test_eq(r.status_code, 200)
print(r.text)

In [ ]:
def g(req, this:Starlette, a:str, b:HttpHeader, state): ...

async def f(req):
    a = await _wrap_req(req, _params(g))
    return Response(str(a))

client = TestClient(Starlette(routes=[Route('/', f, methods=['POST'])]))
response = client.post('/?a=1', data=d)
print(response.text)

In [ ]:
def g(req, this:Starlette, a:str, b:HttpHeader): ...

async def f(req):
    a = await _wrap_req(req, _params(g))
    return Response(str(a))

client = TestClient(Starlette(routes=[Route('/', f, methods=['POST'])]))
response = client.post('/?a=1', data=d)
print(response.text)

In [ ]:
class Bar(BasicRepr):
    @classmethod
    def __from_request__(cls, accept:str=None): return cls(accept)
    def __init__(self, accept): self.accept = accept

def g(b:Bar): ...

async def f(req):
    result = await _wrap_req(req, _params(g))
    return Response(repr(result['b']))

client = TestClient(Starlette(routes=[Route('/', f, methods=['POST'])]))
client.post('/', headers={'Accept': 'application/json'}).text

**Missing Request Params**

If a request param has a default value (e.g. `a:str=''`), the request is valid even if the user doesn't include the param in their request.

In [ ]:
def g(req, this:Starlette, a:str=''): ...

async def f(req):
    a = await _wrap_req(req, _params(g))
    return Response(str(a))

client = TestClient(Starlette(routes=[Route('/', f, methods=['POST'])]))
response = client.post('/', json={}) # no param in request
print(response.text)

If we remove the default value and re-run the request, we should get the following error `Missing required field: a`.

In [ ]:
def g(req, this:Starlette, a:str): ...

async def f(req):
    a = await _wrap_req(req, _params(g))
    return Response(str(a))

client = TestClient(Starlette(routes=[Route('/', f, methods=['POST'])]))
response = client.post('/', json={}) # no param in request
print(response.text)

In [ ]:
#| export
def flat_xt(lst):
    "Flatten lists"
    result = []
    if isinstance(lst,(FT,str)): lst=[lst]
    for item in lst:
        if isinstance(item, (list,tuple)): result.extend(item)
        else: result.append(item)
    return tuple(result)

In [ ]:
x = ft('a',1)
test_eq(flat_xt([x, x, [x,x]]), (x,)*4)
test_eq(flat_xt(x), (x,))

In [ ]:
#| export
class Beforeware:
    "Wrap a `before` function with `skip` path patterns that bypass it"
    def __init__(self, f, skip=None): self.f,self.skip = f,skip or []
    def __repr__(self): return f'Beforeware({self.f}, skip={self.skip})'

In [ ]:
#| export
async def _handle(f, /, *args, **kwargs):
    return (await f(*args, **kwargs)) if is_async_callable(f) else await run_in_threadpool(f, *args, **kwargs)

In [ ]:
async def _f(f): return f"got {f}"
test_eq(await _handle(_f, f='hello'), 'got hello')

## Websockets / SSE

In [ ]:
#| export
async def _wrap_ws(ws, data, params):
    raw = data.pop('HEADERS', {}) or (data.pop('headers') if isinstance(data.get('headers'), dict) else {})  # htmx v4 sends lowercase `headers`
    hdrs = Headers({k.lower():v for k,v in raw.items() if v is not None})
    return await _find_ps(ws, data, hdrs, params)

In [ ]:
#| export
async def _send_ws(ws, resp):
    if not resp: return
    res = to_xml(resp, indent=fh_cfg.indent)
    await ws.send_text(res)

def _ws_endp(recv, conn=None, disconn=None):
    cls = type('WS_Endp', (WebSocketEndpoint,), {"encoding":"text"})

    async def _generic_handler(handler, ws, data=None):
        try:
            wd = await _wrap_ws(ws, loads(data) if data else {}, _params(handler))
            resp = await _handle(handler, **wd)
            if resp: await _send_ws(ws, resp)
        except ValueError as e: await ws.send_text(str(e))

    async def _connect(self, ws):
        await ws.accept()
        await _generic_handler(conn, ws)

    async def _disconnect(self, ws, close_code): await _generic_handler(disconn, ws)
    async def _recv(self, ws, data): await _generic_handler(recv, ws, data)

    if    conn: cls.on_connect    = _connect
    if disconn: cls.on_disconnect = _disconnect
    cls.on_receive = _recv
    return cls

In [ ]:
def on_receive(msg:str): return f"Message text was: {msg}"
c = _ws_endp(on_receive)
cli = TestClient(Starlette(routes=[WebSocketRoute('/', _ws_endp(on_receive))]))
with cli.websocket_connect('/') as ws:
    ws.send_text('{"msg":"Hi!"}')
    data = ws.receive_text()
    assert data == 'Message text was: Hi!', data

In [ ]:
def on_receive(hx_trigger:str): return f"trigger: {hx_trigger}"
cli = TestClient(Starlette(routes=[WebSocketRoute('/', _ws_endp(on_receive))]))
with cli.websocket_connect('/') as ws:
    ws.send_text(json.dumps({"HEADERS": {"Hx-Trigger": "my-btn"}}))
    data = ws.receive_text()
    assert data == 'trigger: my-btn', data


In [ ]:
#| export
def EventStream(s):
    "Create a text/event-stream response from `s`"
    return StreamingResponse(s, media_type="text/event-stream")

In [ ]:
#| export
def signal_shutdown():
    "Return an `asyncio.Event` set when uvicorn receives a shutdown signal"
    from uvicorn.main import Server
    event = asyncio.Event()
    @patch
    def handle_exit(self:Server, *args, **kwargs):
        event.set()
        self.force_exit = True
        self._orig_handle_exit(*args, **kwargs)
    return event

## Routing and application

In [ ]:
#| export
def uri(_arg, **kwargs):
    "Create a URI by URL-encoding `_arg` and appending query parameters from `kwargs`"
    return f"{quote(_arg)}/{urlencode(kwargs, doseq=True)}"

In [ ]:
#| export
def decode_uri(s):
    "Decode a URI created by `uri()` back into argument and keyword dict"
    arg,_,kw = s.partition('/')
    return unquote(arg), {k:v[0] for k,v in parse_qs(kw).items()}

In [ ]:
#| export
from starlette.convertors import StringConvertor

In [ ]:
#| export
StringConvertor.regex = "[^/]*"  # `+` replaced with `*`

@patch
def to_string(self:StringConvertor, value: str) -> str:
    value = str(value)
    assert "/" not in value, "May not contain path separators"
    # assert value, "Must not be empty"  # line removed due to errors
    return value

In [ ]:
#| export
@patch
def url_path_for(self:HTTPConnection, name: str, **path_params):
    lp = self.scope['app'].url_path_for(name, **path_params)
    return URLPath(f"{self.scope['root_path']}{lp}", lp.protocol, lp.host)

In [ ]:
#| export
_verbs = dict(get='hx-get', post='hx-post', put='hx-put', delete='hx-delete', patch='hx-patch', link='href')

def _url_for(req, t):
    "Generate URL for route `t` using request `req`"
    if callable(t): t = t.__routename__
    kw = {}
    if t.find('/')>-1 and (t.find('?')<0 or t.find('/')<t.find('?')): t,kw = decode_uri(t)
    t,m,q = t.partition('?')
    return f"{req.url_path_for(t, **kw)}{m}{q}"

def _find_targets(req, resp):
    "Find and convert route targets in response attributes to URLs"
    if isinstance(resp, tuple):
        for o in resp: _find_targets(req, o)
    if isinstance(resp, FT):
        for o in resp.children: _find_targets(req, o)
        for k,v in _verbs.items():
            t = resp.attrs.pop(k, None)
            if t: resp.attrs[v] = _url_for(req, t)

def _to_xml(req, resp, indent):
    "Convert response to XML string with target URL resolution"
    _find_targets(req, resp)
    return to_xml(resp, indent=indent)

In [ ]:
#| export
_iter_typs = (tuple,list,map,filter,range,types.GeneratorType)

In [ ]:
#| export
def flat_tuple(o):
    "Flatten nested iterables into a single tuple"
    result = []
    if not isinstance(o,_iter_typs): o=[o]
    o = list(o)
    for item in o:
        if isinstance(item, _iter_typs): result.extend(list(item))
        else: result.append(item)
    return tuple(result)

In [ ]:
#| export
def noop_body(c, req):
    "Default Body wrap function which just returns the content"
    return c

In [ ]:
#| export
def respond(req, heads, bdy):
    "Default FT response creation function"
    body_wrap = getattr(req, 'body_wrap', noop_body)
    params = inspect.signature(body_wrap).parameters
    bw_args = (bdy, req) if len(params)>1 else (bdy,)
    body = Body(body_wrap(*bw_args), *flat_xt(req.ftrs), **req.bodykw)
    return Html(Head(*heads, *flat_xt(req.hdrs)), body, **req.htmlkw)

Render fragment if `HX-Request` header is _present_ and
  `HX-History-Restore-Request` header is _absent._

In [ ]:
#| export
def is_full_page(req, resp):
    "Check if response should be rendered as full page or fragment"
    if resp and any(getattr(o, 'tag', '')=='html' for o in resp): return True
    return 'hx-request' in req.headers and 'hx-history-restore-request' not in req.headers

In [ ]:
#| export
def _part_resp(req, resp):
    "Partition response into HTTP headers, background tasks, and content"
    resp = flat_tuple(resp)
    resp = resp + tuple(getattr(req, 'injects', ()))
    http_hdrs,resp = partition(resp, risinstance(HttpHeader))
    tasks,resp = partition(resp, risinstance(BackgroundTask))
    kw = {"headers": {"vary": "HX-Request, HX-History-Restore-Request"}}
    if http_hdrs: kw['headers'] |= {o.k:str(o.v) for o in http_hdrs}
    if tasks:
        ts = BackgroundTasks()
        for t in tasks: ts.tasks.append(t)
        kw['background'] = ts
    resp = tuple(resp)
    if len(resp)==1: resp = resp[0]
    return resp,kw

In [ ]:
#| export
def _canonical(req):
    if not req.app.canonical: return []
    url = str(getattr(req, 'canonical', req.url)).replace('http://', 'https://', 1)
    return [Link(rel="canonical", href=url)]

def _xt_cts(req, resp):
    "Extract content and headers, render as full page or fragment"
    hdr_tags = 'title','meta','link','style','base'
    resp = tuplify(resp)
    heads,bdy = partition(resp, lambda o: getattr(o, 'tag', '') in hdr_tags)
    if not is_full_page(req, resp):
        title = [] if any(getattr(o, 'tag', '')=='title' for o in heads) else [Title(req.app.title)]
        resp = respond(req, [*heads, *title, *_canonical(req)], bdy)
    return _to_xml(req, resp, indent=fh_cfg.indent)

In [ ]:
#| export
def _is_ft_resp(resp):
    "Check if response is a FastTag-compatible type"
    return isinstance(resp, _iter_typs+(HttpHeader,FT)) or hasattr(resp, '__ft__')

In [ ]:
#| export
def _resp(req, resp, cls=empty, status_code=200):
    "Create appropriate HTTP response from request and response data"
    if resp is None: resp=''
    if hasattr(resp, '__response__'): resp = resp.__response__(req)
    if not (isinstance(cls, type) and issubclass(cls, Response)): cls=empty
    if isinstance(resp, FileResponse) and not os.path.exists(resp.path): raise HTTPException(404, resp.path)
    resp,kw = _part_resp(req, resp)
    if isinstance(resp, Response): return resp
    if cls is not empty: return cls(resp, status_code=status_code, **kw)
    if _is_ft_resp(resp):
        cts = _xt_cts(req, resp)
        return HTMLResponse(cts, status_code=status_code, **kw)
    if isinstance(resp, str): cls = HTMLResponse
    elif isinstance(resp, Mapping): cls = JSONResponse
    else:
        resp = str(resp)
        cls = HTMLResponse
    return cls(resp, status_code=status_code, **kw)

In [ ]:
#| export
class Redirect:
    "Use HTMX or Starlette RedirectResponse as required to redirect to `loc`"
    def __init__(self, loc): self.loc = loc
    def __response__(self, req):
        if 'hx-request' in req.headers: return HtmxResponseHeaders(redirect=self.loc)
        return RedirectResponse(self.loc, status_code=303)

In [ ]:
#| export
async def _wrap_call(f, req, params):
    "Wrap function call with request parameter injection"
    wreq = await _wrap_req(req, params)
    return await _handle(f, **wreq)

In [ ]:
#| export
# chkstyle: skip
htmx_exts = {
    "morph": "https://cdn.jsdelivr.net/npm/idiomorph@0.7.3/dist/idiomorph-ext.min.js",
    "head-support": "https://cdn.jsdelivr.net/npm/htmx-ext-head-support@2.0.4/head-support.js",
    "preload": "https://cdn.jsdelivr.net/npm/htmx-ext-preload@2.1.1/preload.js",
    "class-tools": "https://cdn.jsdelivr.net/npm/htmx-ext-class-tools@2.0.1/class-tools.js",
    "loading-states": "https://cdn.jsdelivr.net/npm/htmx-ext-loading-states@2.0.1/loading-states.js",
    "multi-swap": "https://cdn.jsdelivr.net/npm/htmx-ext-multi-swap@2.0.0/multi-swap.js",
    "path-deps": "https://cdn.jsdelivr.net/npm/htmx-ext-path-deps@2.0.0/path-deps.js",
    "remove-me": "https://cdn.jsdelivr.net/npm/htmx-ext-remove-me@2.0.0/remove-me.js",
    "debug": "https://unpkg.com/htmx.org@1.9.12/dist/ext/debug.js",
    "ws": "https://cdn.jsdelivr.net/npm/htmx-ext-ws@2.0.3/ws.js",
    "ws4": "https://cdn.jsdelivr.net/npm/htmx.org@4.0.1/dist/ext/hx-ws.min.js",
    "chunked-transfer": "https://cdn.jsdelivr.net/npm/htmx-ext-transfer-encoding-chunked@0.4.0/transfer-encoding-chunked.js",
    "sse4": "https://cdn.jsdelivr.net/npm/htmx.org@4.0.1/dist/ext/hx-sse.min.js",
    "live": "https://cdn.jsdelivr.net/npm/htmx.org@4.0.1/dist/ext/hx-live.min.js",
    "multipart": "https://cdn.jsdelivr.net/npm/htmx.org@4.0.1/dist/ext/hx-multipart.min.js"
}

The FastHTML `exts` param supports the following:

In [ ]:
print(' '.join(htmx_exts))

In [ ]:
#| export
htmxsrc   = Script(src="https://cdn.jsdelivr.net/npm/htmx.org@2.0.7/dist/htmx.js")
htmx4src  = Script(src="https://cdn.jsdelivr.net/npm/htmx.org@4.0.1/dist/htmx.min.js")
fhjsscr   = Script(src="https://cdn.jsdelivr.net/gh/answerdotai/fasthtml-js@1.0.13/fasthtml.js")
surrsrc   = Script(src="https://cdn.jsdelivr.net/gh/answerdotai/surreal@main/surreal.js")
scopesrc  = Script(src="https://cdn.jsdelivr.net/gh/gnat/css-scope-inline@main/script.js")
viewport  = Meta(name="viewport", content="width=device-width, initial-scale=1, viewport-fit=cover")
charset   = Meta(charset="utf-8")

In [ ]:
#| export
def get_key(key=None, fname='.sesskey'):
    "Get session key from `key` param or read/create from file `fname`"
    if key: return key
    fname = Path(fname)
    if fname.exists(): return fname.read_text()
    key = str(uuid.uuid4())
    fname.write_text(key)
    return key

In [ ]:
#| export
def _list(o):
    "Wrap non-list item in a list, returning empty list if None"
    return [] if not o else list(o) if isinstance(o, (tuple,list)) else [o]

In [ ]:
#| export
def _wrap_ex(f, status_code, hdrs, ftrs, htmlkw, bodykw, body_wrap):
    "Wrap exception handler with FastHTML request processing"
    async def _f(req, exc):
        req.hdrs,req.ftrs,req.htmlkw,req.bodykw = map(deepcopy, (hdrs, ftrs, htmlkw, bodykw))
        req.body_wrap = body_wrap
        res = await _handle(f, req, exc)
        return _resp(req, res, status_code=status_code)
    return _f

In [ ]:
#| export
def qp(p:str, **kw) -> str:
    "Add parameters kw to path p"
    def _sub(m):
        pre,post = m.groups()
        if pre not in kw: return f'{{{pre}{post or ""}}}'
        pre = kw.pop(pre)
        return '' if pre in (False,None) else str(pre)
    p = re.sub(r'\{([^:}]+)(:.+?)?}', _sub, p)
    # encode query params
    return p + ('?' + urlencode({k:'' if v in (False,None) else v for k,v in kw.items()},doseq=True) if kw else '')

`qp` adds query parameters to route path strings

In [ ]:
vals = dict(a=5, b=False, c=[1,2], d='bar', e=None, ab=42)

In [ ]:
res = qp('/foo', **vals)
test_eq(res, '/foo?a=5&b=&c=1&c=2&d=bar&e=&ab=42')

`qp` checks to see if each param should be sent as a query parameter or as part of the route, and encodes that properly.

In [ ]:
path = '/foo/{a}/{d}/{ab:int}'
res = qp(path, **vals)
test_eq(res, '/foo/5/bar/42?b=&c=1&c=2&e=')

In [ ]:
#| export
def def_hdrs(htmx=True, htmx4=False, surreal=True):
    "Default headers for a FastHTML app"
    hdrs = []
    if surreal: hdrs = [surrsrc,scopesrc] + hdrs
    if htmx and htmx4:
        # metaCharacter="-" makes htmx4 use dashes instead of colons
        meta_cfg = Meta(name="htmx-config", content=json.dumps({"metaCharacter": "-"}))
        hdrs = [meta_cfg, htmx4src,fhjsscr] + hdrs 
        # TODO: Check if fhjsscr works with htmx4
    elif htmx: hdrs = [htmxsrc,fhjsscr] + hdrs
    return [charset, viewport] + hdrs

In [ ]:
#| export
cors_allow = Middleware(CORSMiddleware, allow_credentials=True, allow_origins=['*'], allow_methods=['*'], allow_headers=['*'])

iframe_scr = Script(NotStr("""
    function sendmsg() {
        window.parent.postMessage({height: document.documentElement.offsetHeight}, '*');
    }

    window.onload = function() {
        sendmsg();

        const ver = window.htmx?.version || '';
        const isV4 = ver.startsWith('4.');
        const mc = window.htmx?.config?.metaCharacter || ':';

        if (isV4) {
            document.body.addEventListener(`htmx${mc}after${mc}swap`, sendmsg);
            document.body.addEventListener(`htmx${mc}after${mc}ws${mc}message`, sendmsg);
        } else {
            document.body.addEventListener('htmx:afterSettle', sendmsg);
            document.body.addEventListener('htmx:wsAfterMessage', sendmsg);
        }
    };
"""))


In [ ]:
#| export
class _LifespanCtx:
    def __init__(self, gen): self.gen = gen
    async def __aenter__(self): return await anext(self.gen)
    async def __aexit__(self, *_):
        try: await anext(self.gen)
        except StopAsyncIteration: pass
    def __aiter__(self): return self.gen
    async def __anext__(self): return await self.gen.__anext__()

class Lifespan:
    "Combine `on_startup`/`on_shutdown` callbacks and an optional lifespan generator into one context manager"
    def __init__(self, startup=None, shutdown=None, ls=None):
        startup,shutdown = listify(startup),listify(shutdown)
        store_attr()

    def __call__(self, app): return _LifespanCtx(self._run(app))

    async def _run(self, app):
        for f in self.startup: await _handle(f)
        if self.ls:
            async for state in self.ls(app): yield state
        else: yield
        for f in self.shutdown: await _handle(f)

    def on_event(self, event_type): return lambda f: (getattr(self, event_type)).append(f)

In [ ]:
#| export
class FastHTML(Starlette):
    "An HTML-first Starlette app: handler params filled from the request, FT returns rendered as pages or HTMX fragments"
    def __init__(self, debug=False, routes=None, middleware=None, title: str = "FastHTML page", exception_handlers=None,
        on_startup=None, on_shutdown=None, lifespan=None, hdrs=None, ftrs=None, exts=None,
        before=None, after=None, surreal=True, htmx=True, htmx4=False, default_hdrs=True, sess_cls=SessionMiddleware,
        secret_key=None, session_cookie='session_', max_age=365*24*3600, sess_path='/',
        same_site='lax', sess_https_only=False, sess_domain=None, key_fname='.sesskey',
        body_wrap=noop_body, htmlkw=None, nb_hdrs=False, canonical=True, max_part_size=DEF_MAXPART, **bodykw):
        middleware,before,after = map(_list, (middleware,before,after))
        self.htmx4,self.title,self.canonical,self.session_cookie,self.key_fname = htmx4,title,canonical,session_cookie,key_fname
        hdrs,ftrs,exts = map(listify, (hdrs,ftrs,exts))
        if htmx4 and exts: exts = ['ws4' if e in ('ws', 'ws4') else e for e in exts]
        exts = {k:htmx_exts[k] for k in exts}
        htmlkw = htmlkw or {}
        if default_hdrs: hdrs = def_hdrs(htmx, htmx4, surreal=surreal) + hdrs
        hdrs += [Script(src=ext) for ext in exts.values()]
        if IN_NOTEBOOK:
            hdrs.append(iframe_scr) # TODO: check iframe_scr work with htmx4
            from IPython.display import display,HTML
            if nb_hdrs: display(HTML(to_xml(tuple(hdrs))))
            middleware.append(cors_allow)
        self.lifespan = Lifespan(on_startup, on_shutdown, lifespan)
        self.hdrs,self.ftrs = hdrs,ftrs
        self.body_wrap,self.before,self.after,self.htmlkw,self.bodykw,self.max_part_size = body_wrap,before,after,htmlkw,bodykw,max_part_size
        self.secret_key = get_key(secret_key, key_fname)
        if sess_cls:
            sess = Middleware(sess_cls, secret_key=self.secret_key, session_cookie=session_cookie, max_age=max_age, path=sess_path,
                same_site=same_site, https_only=sess_https_only, domain=sess_domain)
            middleware.append(sess)
        exception_handlers = ifnone(exception_handlers, {})
        if 404 not in exception_handlers:
            def _not_found(req, exc): return Response('404 Not Found', status_code=404)
            exception_handlers[404] = _not_found
        excs = {k:_wrap_ex(v, k, hdrs, ftrs, htmlkw, bodykw, body_wrap=body_wrap) for k,v in exception_handlers.items()}
        super().__init__(debug, routes, middleware=middleware, exception_handlers=excs, lifespan=self.lifespan)

    def on_event(self, event_type): return self.lifespan.on_event(event_type)

### Routes

In [ ]:
#| export
class HostRoute(Route):
    "Route with optional host-header constraint using Starlette's {param} pattern syntax"
    def __init__(self, path, endpoint, *, host=None, **kwargs):
        super().__init__(path, endpoint, **kwargs)
        self.host = host
        if host: self.host_regex, self.host_format, self.host_convertors = compile_path(host)

    def matches(self, scope):
        if self.host:
            host = Headers(scope=scope).get("host", "").split(":")[0]
            if not self.host_regex.match(host): return Match.NONE, {}
        return super().matches(scope)

    def __repr__(self) -> str:
        methods = sorted(self.methods or [])
        return f"{self.__class__.__name__}(path={self.path!r}, name={self.name!r}, methods={methods!r}, host={self.host!r})"

In [ ]:
app = FastHTML()
app.router.routes = []

app.router.routes.append(HostRoute('/', lambda _: Response('auth page'), host='auth.{parentdom}', methods=['GET'], name='auth_home'))
app.router.routes.append(HostRoute('/', lambda _: Response('main page'), methods=['GET'], name='main_home'))

cli = TestClient(app)
test_eq(cli.get('/', headers={'host': 'auth.localhost'}).text, 'auth page')
test_eq(cli.get('/', headers={'host': 'localhost'}).text, 'main page')
test_eq(cli.get('/', headers={'host': 'auth.example.com'}).text, 'auth page')
print('all passed')

In [ ]:
app.router.routes.append(HostRoute('/hello', lambda _: Response('api hello'), host='api.{parentdom}', methods=['GET'], name='api_hello'))
app.router.routes.append(HostRoute('/hello', lambda _: Response('normal hello'), methods=['GET'], name='hello'))

test_eq(cli.get('/hello', headers={'host': 'api.mysite.com'}).text, 'api hello')
test_eq(cli.get('/hello', headers={'host': 'mysite.com'}).text, 'normal hello')
test_eq(cli.get('/hello', headers={'host': 'other.mysite.com'}).text, 'normal hello')
print('all passed')

In [ ]:
#| export
@patch
def add_route(self:FastHTML, route):
    "Add or replace a route in the FastHTML app"
    route.methods = [m.upper() for m in listify(route.methods)]
    self.router.routes = [r for r in self.router.routes if not
        (r.path==route.path and r.name == route.name and
            ((route.methods is None) or (set(r.methods) == set(route.methods))))]
    self.router.routes.append(route)
    self.router.routes.sort(key=lambda r: not getattr(r, 'host', None))


In [ ]:
#| export
all_meths = 'get post put delete patch head trace options'.split()

In [ ]:
#| export
@patch
def _endp(self:FastHTML, f, body_wrap, before:Optional[Callable|tuple]=None):
    "Create endpoint wrapper with before/after middleware processing"
    sig = signature_ex(f, True)
    for n,p in sig.parameters.items(): (msg:=_check_anno(n,p.annotation)) and warn(msg)
    async def _f(req):
        resp = None
        req.injects = []
        req.max_part_size = self.max_part_size
        req.hdrs,req.ftrs,req.htmlkw,req.bodykw = map(deepcopy, (self.hdrs,self.ftrs,self.htmlkw,self.bodykw))
        req.hdrs,req.ftrs = listify(req.hdrs),listify(req.ftrs)
        for b in self.before:
            if not resp:
                if isinstance(b, Beforeware): bf,skip = b.f,b.skip
                else: bf,skip = b,[]
                if not any(re.fullmatch(r, req.url.path) for r in skip): resp = await _wrap_call(bf, req, _params(bf))
        for b in listify(before):
            if not resp: resp = await _wrap_call(b, req, _params(b))
        req.body_wrap = body_wrap
        if not resp: resp = await _wrap_call(f, req, sig.parameters)
        for a in self.after:
            wreq = await _wrap_req(req, _params(a))
            wreq['resp'] = resp
            nr = a(**wreq)
            if nr: resp = nr
        return _resp(req, resp, sig.return_annotation)
    return _f

In [ ]:
#| export
@patch
def _add_ws(self:FastHTML, func, path, conn, disconn, name, middleware):
    "Add websocket route to FastHTML app"
    endp = _ws_endp(func, conn, disconn)
    route = WebSocketRoute(path, endpoint=endp, name=name, middleware=middleware)
    route.methods = ['ws']
    self.add_route(route)
    return func

In [ ]:
#| export
@patch
def ws(self:FastHTML, path:str, conn=None, disconn=None, name=None, middleware=None):
    "Add a websocket route at `path`"
    def f(func=noop): return self._add_ws(func, path, conn, disconn, name=name, middleware=middleware)
    return f

In [ ]:
#|export
@patch
def add_websocket_route(self:FastHTML, path, func, conn=None, disconn=None, name=None, middleware=None):
    "Add a websocket route at `path` (Starlette-compatible API)"
    return self._add_ws(func, path, conn, disconn, name=name, middleware=middleware)

In [ ]:
#| export
def _mk_locfunc(f, p, app=None):
    "Create a location function wrapper with route path and to() method"
    class _lf:
        def __init__(self):
            update_wrapper(self, f)
            self.app = app

        def __call__(self, *args, **kw): return f(*args, **kw)
        def to(self, **kw): return qp(p, **kw)
        def __str__(self): return p

    return _lf()

In [ ]:
#| export
def nested_name(f):
    "Get name of function `f` using '_' to join nested function names"
    return f.__qualname__.replace('.<locals>.', '_')

In [ ]:
def f():
    def g(): ...
    return g

In [ ]:
func = f()
nested_name(func)

In [ ]:
#| export
@patch
def _add_routes(self:FastHTML, cls, path, methods, name, include_in_schema, body_wrap, host=None, before:Optional[Callable|tuple]=None):
    "Add HTTP routes from methods on endpoint class `cls`"
    assert not methods, '`methods` is not supported for class route groups; define HTTP methods as class methods instead'
    lf = _mk_locfunc(cls, path, app=self)
    lf.__routename__ = name
    for meth in all_meths:
        handler = getattr(cls, meth, None)
        if handler: self._add_route(handler, path, meth, name, include_in_schema, body_wrap, host=host, before=before)
    return lf

In [ ]:
#| export
def _route_pn(func, path, name):
    "Infer route name/function name/path from an endpoint or endpoint class"
    fn = nested_name(func)
    if isinstance(func, type): fn = fn[0].lower()+fn[1:]
    p = None if callable(path) else path
    if not name: name = fn
    if not p: p = '/'+('' if fn=='index' else fn)
    return name,fn,p

@patch
def _add_route(self:FastHTML, func, path, methods, name, include_in_schema, body_wrap, host=None, before:Optional[Callable|tuple]=None):
    "Add HTTP route to FastHTML app with automatic method detection"
    n,fn,p = _route_pn(func, path, name)
    if isinstance(func, type): return self._add_routes(func, p, methods, n, include_in_schema, body_wrap, host=host, before=before)
    if methods: m = [methods] if isinstance(methods,str) else methods
    elif fn in all_meths and p is not None: m = [fn]
    else: m = ['get','post']
    endp = self._endp(func, body_wrap or self.body_wrap, before=before)
    route = HostRoute(p, endpoint=endp, methods=m, name=n, include_in_schema=include_in_schema, host=host)
    self.add_route(route)
    lf = _mk_locfunc(func, p, app=self)
    lf.__routename__ = n
    return lf

In [ ]:
#| export
@patch
def route(self:FastHTML, path:str=None, methods=None, name=None, include_in_schema=True, body_wrap=None, host=None, before:Optional[Callable|tuple]=None):
    "Add a route at `path`"
    def f(func):
        return self._add_route(func, path, methods, name=name, include_in_schema=include_in_schema, body_wrap=body_wrap, host=host, before=before)
    return f(path) if callable(path) else f

for o in all_meths: setattr(FastHTML, o, partialmethod(FastHTML.route, methods=o))

In [ ]:
app = FastHTML()
@app.get
def foo(a:str, b:list[int]): ...

foo.to(a='bar', b=[1,2])

In [ ]:
@app.get('/foo/{a}')
def foo(a:str, b:list[int]): ...

foo.to(a='bar', b=[1,2])

### Utils

In [ ]:
#| export
@patch
def set_lifespan(self:FastHTML, value):
    "Set the lifespan context manager for the FastHTML app"
    if inspect.isasyncgenfunction(value): value = contextlib.asynccontextmanager(value)
    self.router.lifespan_context = value

In [ ]:
#| export
def serve(
    appname=None, # Name of the module
    app='app', # App instance to be served
    host='0.0.0.0', # If host is 0.0.0.0 will convert to localhost
    port=None, # If port is None it will default to 5001 or the PORT environment variable
    reload=True, # Default is to reload the app upon code changes
    procs=None, # Processes to run on consecutive ports from `port`; default 1 or the PROCS environment variable
    **kwargs
    ):
    "Run the app in an async server, with live reload set as the default."
    from uvicorn import run
    bk = inspect.currentframe().f_back
    glb,back = bk.f_globals,bk.f_back
    if not appname:
        if glb.get('__name__')=='__main__': appname = Path(glb.get('__file__', '')).stem
        elif back and back.f_globals.get('__name__')=='__main__': appname = inspect.getmodule(bk).__name__
    if appname:
        if not port: port=int(os.getenv("PORT", default=5001))
        if not procs: procs=int(os.getenv("PROCS", default=1))
        if procs>1 and 'FH_PROCNUM' not in os.environ: return _spawn(procs)
        port += int(os.getenv('FH_PROCNUM', 0))
        link = f'http://{"localhost" if host=="0.0.0.0" else host}:{port}'
        print('Link: '+ S.light_red.bold(link))
        run(f'{appname}:{app}', host=host, port=port, reload=reload, **kwargs)

`procs` runs the app as that many processes on consecutive ports from `port`, each with its own reload supervisor, and defaults to the `PROCS` environment variable. `serve` re-executes the script once per process with `FH_PROCNUM` set to its number, which is also the guard against a child spawning again. An app reads `FH_PROCNUM` for work that only one process should do, such as a background job: it is `'0'` in the first process, and unset in an ordinary single-process run. A reverse proxy in front of the ports decides how requests reach them.

In [ ]:
#| export
def _spawn(n):
    "Run this script `n` times over, numbering each copy in `FH_PROCNUM`, and wait for them all"
    ps = [subprocess.Popen([sys.executable, *sys.argv], env={**os.environ, 'FH_PROCNUM':str(i)}) for i in range(n)]
    signal.signal(signal.SIGTERM, lambda *a: [p.terminate() for p in ps])
    for p in ps: p.wait()

Starlette does not cancel a running handler when a client disconnects: the disconnect is only visible as an `http.disconnect` message on the ASGI receive channel, which ordinary handlers never read. `until_disconnect` races a coroutine against that message, cancelling the work if the client goes away first.

In [ ]:
#| export
async def _wait_disconnect(req):
    while (await req.receive())["type"] != "http.disconnect": pass

async def until_disconnect(req, coro):
    "Await `coro`, cancelling it (and returning an empty response) if the client disconnects first"
    work,watch = asyncio.ensure_future(coro),asyncio.ensure_future(_wait_disconnect(req))
    try:
        done,_ = await asyncio.wait({work,watch}, return_when=asyncio.FIRST_COMPLETED)
        if work in done: return work.result()
        return Response(status_code=204)
    finally:
        watch.cancel()
        if not work.done(): work.cancel()

`cancel_on_disconnect` applies this to a route handler. `splice_sig` adds a private `Request`-annotated parameter to the exposed signature, which FastHTML fills with the connection, so the handler's own parameters are unchanged. It requires an async handler, since a sync handler runs in a threadpool and cannot be cancelled.

In [ ]:
#| export
def cancel_on_disconnect(
    f=None, # Async route handler to wrap
    *,
    timeout=None, # Also cancel the handler after this many seconds
):
    "Route decorator: cancel the handler when the client disconnects"
    if f is None: return partial(cancel_on_disconnect, timeout=timeout)
    assert is_async_callable(f), "cancel_on_disconnect requires an async handler"
    async def _f(__req: Request, *args, **kw):
        c = until_disconnect(__req, f(**kw))
        return await (asyncio.wait_for(c, timeout) if timeout is not None else c)
    return splice_sig(_f, f)

In [ ]:
#| export
class Client:
    "A simple httpx ASGI client that doesn't require `async`"
    def __init__(self, app, url="http://testserver"):
        import httpx2
        self.cli = httpx2.AsyncClient(transport=httpx2.ASGITransport(app), base_url=url)

    def _sync(self, method, url, **kwargs):
        async def _request(): return await self.cli.request(method, url, **kwargs)
        with from_thread.start_blocking_portal() as portal: return portal.call(_request)

for o in ('get', 'post', 'delete', 'put', 'patch', 'options'): setattr(Client, o, partialmethod(Client._sync, o))

In [ ]:
app = FastHTML(routes=[Route('/', lambda _: Response('test'))])
cli = Client(app)

cli.get('/').text

Note that you can also use Starlette's `TestClient` instead of FastHTML's `Client`. They should be largely interchangable.

## FastHTML Tests

In [ ]:
def get_cli(app): return app,TestClient(app),app.route

In [ ]:
app,cli,rt = get_cli(FastHTML(secret_key='soopersecret'))

In [ ]:
app,cli,rt = get_cli(FastHTML(title="My Custom Title"))
@app.get
def foo(): return Div("Hello World")

print(app.routes)

response = cli.get('/foo')
assert '<title>My Custom Title</title>' in response.text

foo.to(param='value')

In [ ]:
@rt(host='test.foo')
def foo(): return Div("Goodbye World")

print(app.routes)

assert "Goodbye World" in cli.get('/foo', headers={'host': 'test.foo'}).text
assert "Hello World" in cli.get('/foo').text

In [ ]:
response = cli.get('/foo')
assert '<title>My Custom Title</title>' in response.text

foo.to(param='value')

In [ ]:
app,cli,rt = get_cli(FastHTML())

@rt('/xt2')
def get(): return H1('bar')

txt = cli.get('/xt2').text
assert '<title>FastHTML page</title>' in txt and '<h1>bar</h1>' in txt and '<html>' in txt

In [ ]:
def test_r(cli, path, exp, meth='get', hx=False, **kwargs):
    if hx: kwargs['headers'] = {'hx-request':"1"}
    test_eq(getattr(cli, meth)(path, **kwargs).text, exp)

In [ ]:
@dataclass
class Card: id:str; title:str

@rt('/class-cards/{id}')
class CardMethods:
    async def get(id:str): return f'get {id}'
    async def patch(card:Card): return asdict(card)
    async def delete(id:str): return f'delete {id}'

test_eq(cli.get('/class-cards/abc').text, 'get abc')
test_r(cli, '/class-cards/abc', '{"id":"abc","title":"Foo"}', 'patch', data={'title':'Foo'})
test_eq(cli.delete('/class-cards/abc').text, 'delete abc')
test_eq(CardMethods.to(id='xyz'), '/class-cards/xyz')

In [ ]:
@rt
class Things:
    def get(id:str): return f'thing {id}'

test_eq(cli.get('/things?id=7').text, 'thing 7')
test_eq(Things.to(id=7), '/things?id=7')

In [ ]:
@rt("/hi")
def get(): return 'Hi there'

r = cli.get('/hi')
r.text

In [ ]:
@rt("/hi")
def post(): return 'Postal'

cli.post('/hi').text

In [ ]:
@app.get("/hostie")
def show_host(req): return req.headers['host']

cli.get('/hostie').text

In [ ]:
@app.get("/setsess")
def set_sess(session):
   session['foo'] = 'bar'
   return 'ok'

@app.ws("/ws")
def ws(self, msg:str, ws:WebSocket, session): return f"Message text was: {msg} with session {session.get('foo')}, from client: {ws.client}"

cli.get('/setsess')
with cli.websocket_connect('/ws') as ws:
    ws.send_text('{"msg":"Hi!"}')
    data = ws.receive_text()
assert 'Message text was: Hi! with session bar' in data
print(data)

In [ ]:
@rt
def yoyo(): return 'a yoyo'

cli.post('/yoyo').text

In [ ]:
@app.get
def autopost(): return Html(Div('Text.', hx_post=yoyo()))
print(cli.get('/autopost').text)

In [ ]:
@app.get
def autopost2(): return Html(Body(Div('Text.', cls='px-2', hx_post=show_host.to(a='b'))))
print(cli.get('/autopost2').text)

In [ ]:
@app.get
def autoget2(): return Html(Div('Text.', hx_get=show_host))
print(cli.get('/autoget2').text)

In [ ]:
@rt('/user/{nm}', name='gday')
def get(nm:str=''): return f"Good day to you, {nm}!"
cli.get('/user/Alexis').text

In [ ]:
@app.get
def autolink(): return Html(Div('Text.', link=uri('gday', nm='Alexis')))
print(cli.get('/autolink').text)

In [ ]:
@rt('/link')
def get(req): return f"{req.url_for('gday', nm='Alexis')}; {req.url_for('show_host')}"

cli.get('/link').text

In [ ]:
@app.get("/background")
async def background_task(request):
    async def long_running_task():
        await asyncio.sleep(0.1)
        print("Background task completed!")
    return P("Task started"), BackgroundTask(long_running_task)

response = cli.get("/background")

In [ ]:
test_eq(app.router.url_path_for('gday', nm='Jeremy'), '/user/Jeremy')

In [ ]:
hxhdr = {'headers':{'hx-request':"1"}}

@rt('/ft')
def get(): return Title('Foo'),H1('bar')

txt = cli.get('/ft').text
assert '<title>Foo</title>' in txt and '<h1>bar</h1>' in txt and '<html>' in txt

@rt('/xt2')
def get(): return H1('bar')

txt = cli.get('/xt2').text
assert '<title>FastHTML page</title>' in txt and '<h1>bar</h1>' in txt and '<html>' in txt

assert cli.get('/xt2', **hxhdr).text.strip() == '<h1>bar</h1>'

@rt('/xt3')
def get(): return Html(Head(Title('hi')), Body(P('there')))

txt = cli.get('/xt3').text
assert '<title>FastHTML page</title>' not in txt and '<title>hi</title>' in txt and '<p>there</p>' in txt

In [ ]:
# declaration-time warnings
def _noanno(nope): return nope
def _bare(names:list): return str(names)
test_warns(lambda: rt('/oops')(_noanno))
test_warns(lambda: rt('/barelist')(_bare))

In [ ]:
# runtime warnings
test_warns(lambda: cli.get('/oops?nope=1'))
test_warns(lambda: cli.get('/barelist?names=1'))

In [ ]:
ModelName = str_enum('ModelName', "alexnet", "resnet", "lenet")
fake_db = [{"name": "Foo"}, {"name": "Bar"}]

In [ ]:
@rt('/html/{idx}')
async def get(idx:int): return Body(H4(f'Next is {idx+1}.'))

In [ ]:
@rt("/models/{nm}")
def get(nm:ModelName): return nm

@rt("/files/{path}")
async def get(path: Path): return path.with_suffix('.txt')

@rt("/items/")
def get(idx:int|None = 0): return fake_db[idx]

@rt("/idxl/")
def get(idx:list[int]): return str(idx)

@rt("/optidxl/")
def get(idx:Optional[list[int]]): return str(idx)  # chkstyle: ignore

@rt("/unionidxl/")
def get(idx:list[int] | None): return str(idx)

In [ ]:
r = cli.get('/html/1', headers={'hx-request':"1"})
assert '<h4>Next is 2.</h4>' in r.text
test_r(cli, '/models/alexnet', 'alexnet')
test_r(cli, '/files/foo', 'foo.txt')
test_r(cli, '/items/?idx=1', '{"name":"Bar"}')
test_r(cli, '/items/', '{"name":"Foo"}')
assert cli.get('/items/?idx=g').text=='404 Not Found'
assert cli.get('/items/?idx=g').status_code == 404
test_r(cli, '/idxl/?idx=1&idx=2', '[1, 2]')
assert cli.get('/idxl/?idx=1&idx=g').status_code == 404
test_r(cli, '/optidxl/?idx=1&idx=2', '[1, 2]')
test_r(cli, '/unionidxl/?idx=1&idx=2', '[1, 2]')

In [ ]:
app = FastHTML()
rt = app.route
cli = TestClient(app)
@app.route(r'/static/{path:path}.jpg')
def index(path:str): return f'got {path}'
@app.route(r'/static/{path:path}')
def foo(path:str, a:int): return f'also got {path},{a}'
cli.get('/static/sub/a.b.jpg').text

In [ ]:
cli.get('/static/sub/a.b?a=1').text

In [ ]:
app.chk = 'foo'

In [ ]:
@app.get("/booly/")
def _(coming:bool=True): return 'Coming' if coming else 'Not coming'

@app.get("/datie/")
def _(d:parsed_date): return d

@app.get("/ua")
async def _(user_agent:str): return user_agent

@app.get("/hxtest")
def _(htmx): return htmx.request

@app.get("/hxtest2")
def _(foo:HtmxHeaders, req): return foo.request

@app.get('/hxprompt')
def _(hx_prompt:str, htmx:HtmxHeaders): return JSONResponse([hx_prompt, htmx.prompt])

@app.get("/app")
def _(app): return app.chk

@app.get("/app2")
def _(foo:FastHTML): return foo.chk,HttpHeader("mykey", "myval")

@app.get("/app3")
def _(foo:FastHTML): return HtmxResponseHeaders(location="http://example.org")

@app.get("/app4")
def _(foo:FastHTML): return Redirect("http://example.org")

In [ ]:
test_r(cli, '/booly/?coming=true', 'Coming')
test_r(cli, '/booly/?coming=no', 'Not coming')
date_str = "17th of May, 2024, 2p"
test_r(cli, f'/datie/?d={date_str}', '2024-05-17 14:00:00')
test_r(cli, '/ua', 'FastHTML', headers={'User-Agent':'FastHTML'})
test_r(cli, '/hxtest' , '1', headers={'HX-Request':'1'})
test_r(cli, '/hxtest2', '1', headers={'HX-Request':'1'})
test_r(cli, '/app' , 'foo')
for headers, expected in [
    ({'HX-Prompt':'%20 + 100%'}, '%20 + 100%'),
    ({'HX-Prompt':'Hi%20%C3%A9+%2520', 'HX-Request-Type':'partial'}, 'Hi é+%20'),
    ({'HX-Prompt':'Hi%20%C3%A9%2B%2520', 'HX-Prompt-URI-AutoEncoded':'true'}, 'Hi é+%20')]:
    test_eq(cli.get('/hxprompt', headers=headers).json(), [expected, expected])

In [ ]:
r = cli.get('/app2', **hxhdr)
test_eq(r.text, 'foo')
test_eq(r.headers['mykey'], 'myval')

In [ ]:
r = cli.get('/app3')
test_eq(r.headers['HX-Location'], 'http://example.org')

In [ ]:
r = cli.get('/app4', follow_redirects=False)
test_eq(r.status_code, 303)

In [ ]:
r = cli.get('/app4', headers={'HX-Request':'1'})
test_eq(r.headers['HX-Redirect'], 'http://example.org')

In [ ]:
@rt
def meta(): return ((Title('hi'), H1('hi')), (Meta(property='image'), Meta(property='site_name')))

t = cli.post('/meta').text
assert re.search(r'<body>\s*<h1>hi</h1>\s*</body>', t)
assert '<meta' in t

In [ ]:
@app.post('/profile/me')
def profile_update(user_name: str): return user_name

test_r(cli, '/profile/me', 'Alexis', 'post', data={'user-name':'Alexis'})
test_r(cli, '/profile/me?user-name=Jeremy&user-name=Alexis', 'Alexis', 'post')
test_r(cli, '/profile/me', 'Jeremy', 'post', data={'user-name':'Alexis', 'user_name':'Jeremy'})
test_r(cli, '/profile/me', 'Missing required field: user_name', 'post', data={})

In [ ]:
# Example post request with parameter that has a default value
@app.post('/pet/dog')
def pet_dog(dogname: str = None): return dogname

# Working post request with optional parameter
test_r(cli, '/pet/dog', '', 'post', data={})

In [ ]:
@dataclass
class Bodie: a:int;b:str

@rt("/bodie/{nm}")
def post(nm:str, data:Bodie):
    res = asdict(data)
    res['nm'] = nm
    return res

@rt("/cards/{id}")
def put(card:Card): return asdict(card)

@app.post("/bodied/")
def bodied(data:dict): return data

nt = namedtuple('Bodient', ['a','b'])

@app.post("/bodient/")
def bodient(data:nt): return asdict(data)

class BodieTD(TypedDict): a:int;b:str='foo'

@app.post("/bodietd/")
def bodient(data:BodieTD): return data

class Bodie2:
    a_val:int|None; b:str
    def __init__(self, a_val, b='foo'): store_attr()

@rt("/bodie2/", methods=['get','post'])
def bodie(d:Bodie2): return f"a: {d.a_val}; b: {d.b}"

In [ ]:
from fasthtml.xtend import Titled

In [ ]:
d = dict(a=1, b='foo')

test_r(cli, '/bodie/me', '{"a":1,"b":"foo","nm":"me"}', 'post', data=dict(a=1, b='foo', nm='me'))
test_r(cli, '/cards/abc', '{"id":"abc","title":"Foo"}', 'put', data={'title':'Foo'})
test_r(cli, '/bodied/', '{"user-name":"Alexis"}', 'post', data={'user-name':'Alexis'})
test_r(cli, '/bodie2/', 'a: 1; b: foo', 'post', data={'a-val':2, 'a_val':1})
test_r(cli, '/bodie2/?a-val=1&b=foo&nm=me', 'a: 1; b: foo')
test_r(cli, '/bodient/', '{"a":"1","b":"foo"}', 'post', data=d)
test_r(cli, '/bodietd/', '{"a":1,"b":"foo"}', 'post', data=d)

In [ ]:
# Testing POST with Content-Type: application/json
@app.post("/")
def index(it: Bodie): return Titled("It worked!", P(f"{it.a}, {it.b}"))

s = json.dumps({"b": "Lorem", "a": 15})
response = cli.post('/', headers={"Content-Type": "application/json"}, data=s).text
assert "<title>It worked!</title>" in response and "<p>15, Lorem</p>" in response

In [ ]:
# Testing POST with Content-Type: application/json
@app.post("/bodytext")
def index(body): return body

response = cli.post('/bodytext', headers={"Content-Type": "application/json"}, data=s).text
test_eq(response, '{"b": "Lorem", "a": 15}')

In [ ]:
files = [('files', ('file1.txt', b'content1')), ('files', ('file2.txt', b'content2'))]

In [ ]:
@rt("/uploads")
async def post(files:list[UploadFile]): return ','.join([(await file.read()).decode() for file in files])

res = cli.post('/uploads', files=files)
print(res.status_code)
print(res.text)

In [ ]:
res = cli.post('/uploads', files=[files[0]])
test_eq(res.status_code, 200)
test_eq(res.text, 'content1')

In [ ]:
@rt("/setsess")
def get(sess, foo:str=''):
    now = datetime.now()
    sess['auth'] = str(now)
    return f'Set to {now}'

@rt("/getsess")
def get(sess): return f'Session time: {sess["auth"]}'

print(cli.get('/setsess').text)
time.sleep(0.01)

cli.get('/getsess').text

In [ ]:
@rt("/sess-first")
def post(sess, name: str):
    sess["name"] = name
    return str(sess)

cli.post('/sess-first', data={'name': 2})

@rt("/getsess-all")
def get(sess): return sess['name']

test_eq(cli.get('/getsess-all').text, '2')

In [ ]:
@rt("/upload")
async def post(uf:UploadFile): return (await uf.read()).decode()

with open('../../CHANGELOG.md', 'rb') as f: print(cli.post('/upload', files={'uf':f}, data={'msg':'Hello'}).text[:15])

In [ ]:
@rt("/form-submit/{list_id}")
def options(list_id: str):
    headers = {'Access-Control-Allow-Origin': '*', 'Access-Control-Allow-Methods': 'POST', 'Access-Control-Allow-Headers': '*'}
    return Response(status_code=200, headers=headers)

In [ ]:
h = cli.options('/form-submit/2').headers
test_eq(h['Access-Control-Allow-Methods'], 'POST')

In [ ]:
from fasthtml.authmw import user_pwd_auth

In [ ]:
def _not_found(req, exc): return Div('nope')

app,cli,rt = get_cli(FastHTML(exception_handlers={404:_not_found}))

txt = cli.get('/').text
assert '<div>nope</div>' in txt
assert '<!doctype html>' in txt

In [ ]:
app,cli,rt = get_cli(FastHTML())

@rt("/{name}/{age}")
def get(name: str, age: int): return Titled(f"Hello {name.title()}, age {age}")

assert '<title>Hello Uma, age 5</title>' in cli.get('/uma/5').text
assert '404 Not Found' in cli.get('/uma/five').text

In [ ]:
auth = user_pwd_auth(testuser='spycraft')
app,cli,rt = get_cli(FastHTML(middleware=[auth]))

@rt("/locked")
def get(auth): return 'Hello, ' + auth

test_eq(cli.get('/locked').text, 'not authenticated')
test_eq(cli.get('/locked', auth=("testuser","spycraft")).text, 'Hello, testuser')

In [ ]:
auth = user_pwd_auth(testuser='spycraft')
app,cli,rt = get_cli(FastHTML(middleware=[auth]))

@rt("/locked")
def get(auth): return 'Hello, ' + auth

test_eq(cli.get('/locked').text, 'not authenticated')
test_eq(cli.get('/locked', auth=("testuser","spycraft")).text, 'Hello, testuser')

HTMX4 fragment requests receive toast partials. Full pages render the toast container. Set `htmx=False, htmx4=True` when supplying your own HTMX4 script.

In [ ]:
from fasthtml.toaster import *

In [ ]:
app,cli,rt = get_cli(FastHTML(htmx=False, htmx4=True, hdrs=[htmx4src]))
setup_toasts(app)
tm = 'Here is a toast!'
def index(sess):
    add_toast(sess, tm, 'success')
    return P('I love toast!')
rt('/')(index)
page = cli.get('/').text
test_eq(page.count(htmx4src.attrs['src']), 1)
test_eq(page.count(f'id="{tcid}"'), 1)
assert '<hx-partial' not in page
assert '<hx-partial' not in cli.get('/', headers={'hx-request':'1', 'hx-history-restore-request':'1'}).text
r = cli.get('/', **hxhdr)
assert tm in r.text and '<hx-partial ' in r.text and f'hx-target="#{tcid}"' in r.text
assert 'hx-swap="beforeend"' in r.text and 'hx-swap-oob' not in r.text
legacy,legacy_cli,legacy_rt = get_cli(FastHTML())
setup_toasts(legacy)
legacy_rt('/')(index)
r = legacy_cli.get('/', **hxhdr)
assert tm in r.text and 'hx-swap-oob=' in r.text

Here's a quick sanity check to confirm that toasts are rendered in the toast container correctly. See [issue](https://github.com/AnswerDotAI/fasthtml/issues/762).

In [ ]:
assert 'id="fh-toast-container"' in r.text

In [ ]:
app,cli,rt = get_cli(FastHTML(on_startup=[lambda: setattr(app, '_started', True)],
    on_shutdown=[lambda: setattr(app, '_stopped', True)]))

@rt
def index(): return str(app._started)

with cli: test_eq(cli.get('/').text, 'True')
test_eq(app._stopped, True)

In [ ]:
app,cli,rt = get_cli(FastHTML())

@app.on_event('startup')
def _(): app._evt = 'fired'

@rt
def index(): return app._evt

with cli: test_eq(cli.get('/').text, 'fired')

A route-level `before` can set session data, short-circuit with a response, or raise to block access:

In [ ]:
def check_user(sess, user:str=''):
    if user == 'admin':
        sess['foo'] = 'bar'
        return None
    if user == 'guest': return 'guest view'
    raise HTTPException(403, 'forbidden')

@rt('/dashboard', before=check_user)
def get(sess): return sess.get('foo')

test_eq(cli.get('/dashboard?user=admin').text, 'bar')
test_eq(cli.get('/dashboard?user=guest').text, 'guest view')
test_eq(cli.get('/dashboard?user=other').status_code, 403)

Multiple `before` functions run in order, each receiving the injected request params:

In [ ]:
def set_a(sess): sess['a'] = 1
def set_b(sess): sess['b'] = 2

@rt('/multi', before=(set_a, set_b))
def get(sess): return f"{sess.get('a')},{sess.get('b')}"

test_eq(cli.get('/multi').text, '1,2')

A truthy return from any `before` becomes the response; remaining `before`s and the handler are skipped (return `None` to continue):

In [ ]:
calls = []
def first(sess): 
    calls.append('first')
    return 'stop'
def second(sess): calls.append('second')

@rt('/chain', before=(first, second))
def get(sess): return 'handler'

test_eq(cli.get('/chain').text, 'stop')
test_eq(calls, ['first'])

`cancel_on_disconnect` handlers keep their normal parameters; the handler is cancelled if the client disconnects, or after `timeout` seconds if given:

In [ ]:
app,cli,rt = get_cli(FastHTML())

@rt('/cq')
@cancel_on_disconnect
async def cq(q:str): return f'got {q}'

test_eq(cli.get('/cq?q=x').text, 'got x')

In [ ]:
@rt('/sleepy')
@cancel_on_disconnect(timeout=0.01)
async def sleepy():
    await asyncio.sleep(1)
    return 'nope'

test_fail(lambda: cli.get('/sleepy'))

On disconnect the handler is cancelled and an empty response returned, which the server discards since the client is gone. Here we simulate the ASGI `http.disconnect` message directly:

In [ ]:
async def _disc(): return {'type': 'http.disconnect'}
cancelled = []
async def _work():
    try: await asyncio.sleep(10)
    except asyncio.CancelledError:
        cancelled.append(True)
        raise

async def _t():
    scope = dict(type='http', method='GET', path='/', query_string=b'', headers=[])
    res = await until_disconnect(Request(scope, _disc), _work())
    await asyncio.sleep(0.01)
    return res

test_eq(run_sync(_t()).status_code, 204)
test_eq(cancelled, [True])

## APIRouter

In [ ]:
#| export
class RouteFuncs:
    "Attr-access store of named route functions (HTTP verb names excluded)"
    def __init__(self): super().__setattr__('_funcs', {})
    def __setattr__(self, name, value): self._funcs[name] = value
    def __getattr__(self, name):
        if name in all_meths: raise AttributeError("Route functions with HTTP Names are not accessible here")
        try: return self._funcs[name]
        except KeyError: raise AttributeError(f"No route named {name} found in route functions")
    def __dir__(self): return list(self._funcs.keys())

In [ ]:
#| export
class APIRouter:
    "Add routes to an app"
    def __init__(self, prefix:str|None=None, body_wrap=noop_body):
        self.routes,self.wss = [],[]
        self.rt_funcs = RouteFuncs()  # Store wrapped route function for discoverability
        self.prefix = prefix if prefix else ""
        self.body_wrap = body_wrap

    def _wrap_func(self, func, path=None, name=None):
        wrapped = _mk_locfunc(func, path)
        wrapped.__routename__ = name
        # If you are using def get/post/etc method names, this approach is not supported
        if name not in all_meths: setattr(self.rt_funcs, name, wrapped)
        return wrapped

    def __call__(self, path:str=None, methods=None, name=None, include_in_schema=True, body_wrap=None):
        "Add a route at `path`"
        def f(func):
            n,_,p = _route_pn(func, path, name)
            p = self.prefix + p
            wrapped = self._wrap_func(func, p, n)
            self.routes.append((func, p, methods, n, include_in_schema, body_wrap or self.body_wrap))
            return wrapped
        return f(path) if callable(path) else f

    def __getattr__(self, name):
        try: return getattr(self.rt_funcs, name)
        except AttributeError: return super().__getattr__(self, name)

    def to_app(self, app):
        "Add routes to `app`"
        for args in self.routes: app._add_route(*args)
        for args in self.wss: app._add_ws(*args)

    def ws(self, path:str, conn=None, disconn=None, name=None, middleware=None):
        "Add a websocket route at `path`"
        def f(func=noop): return self.wss.append((func, f"{self.prefix}{path}", conn, disconn, name, middleware))
        return f

In [ ]:
ar = APIRouter()

In [ ]:
@ar("/hi")
def get(): return 'Hi there'
@ar("/hi")
def post(): return 'Postal'
@ar
def ho(): return 'Ho ho'
@ar("/hostie")
def show_host(req): return req.headers['host']
@ar
def yoyo(): return 'a yoyo'
@ar
def index(): return "home page"
@ar('/class-cards/{id}')
class CardMethods:
    async def get(id:str): return f'get {id}'
    async def patch(card:Card): return asdict(card)
    async def delete(id:str): return f'delete {id}'


@ar.ws("/ws")
def ws(self, msg:str): return f"Message text was: {msg}"

In [ ]:
app,cli,_ = get_cli(FastHTML())
ar.to_app(app)

In [ ]:
assert str(yoyo) == '/yoyo'
# ensure route functions are properly discoverable on `APIRouter` and `APIRouter.rt_funcs`
assert ar.prefix == ''
assert str(ar.rt_funcs.index) == '/'
assert str(ar.index) == '/'
with ExceptionExpected(): ar.blah()
with ExceptionExpected(): ar.rt_funcs.blah()
# ensure any route functions named using an HTTPMethod are not discoverable via `rt_funcs`
assert "get" not in ar.rt_funcs._funcs.keys()

In [ ]:
test_eq(cli.get('/hi').text, 'Hi there')
test_eq(cli.post('/hi').text, 'Postal')
test_eq(cli.get('/hostie').text, 'testserver')
test_eq(cli.post('/yoyo').text, 'a yoyo')

test_eq(cli.get('/ho').text, 'Ho ho')
test_eq(cli.post('/ho').text, 'Ho ho')

test_eq(cli.get('/class-cards/abc').text, 'get abc')
test_r(cli, '/class-cards/abc', '{"id":"abc","title":"Foo"}', 'patch', data={'title':'Foo'})
test_eq(cli.delete('/class-cards/abc').text, 'delete abc')
test_eq(CardMethods.to(id='xyz'), '/class-cards/xyz')

In [ ]:
with cli.websocket_connect('/ws') as ws:
    ws.send_text('{"msg":"Hi!"}')
    data = ws.receive_text()
    assert data == 'Message text was: Hi!'

In [ ]:
ar2 = APIRouter("/products")

In [ ]:
@ar2("/hi")
def get(): return 'Hi there'
@ar2("/hi")
def post(): return 'Postal'
@ar2
def ho(): return 'Ho ho'
@ar2("/hostie")
def show_host(req): return req.headers['host']
@ar2
def yoyo(): return 'a yoyo'
@ar2
def index(): return "home page"
@ar2
class Products:
    def get(id:str): return f'product {id}'


@ar2.ws("/ws")
def ws(self, msg:str): return f"Message text was: {msg}"

In [ ]:
app,cli,_ = get_cli(FastHTML())
ar2.to_app(app)

In [ ]:
assert str(yoyo) == '/products/yoyo'
assert ar2.prefix == '/products'
assert str(ar2.rt_funcs.index) == '/products/'
assert str(ar2.index) == '/products/'
assert str(ar.index) == '/'
with ExceptionExpected(): ar2.blah()
with ExceptionExpected(): ar2.rt_funcs.blah()
assert "get" not in ar2.rt_funcs._funcs.keys()

In [ ]:
test_eq(cli.get('/products/hi').text, 'Hi there')
test_eq(cli.post('/products/hi').text, 'Postal')
test_eq(cli.get('/products/hostie').text, 'testserver')
test_eq(cli.post('/products/yoyo').text, 'a yoyo')

test_eq(cli.get('/products/ho').text, 'Ho ho')
test_eq(cli.post('/products/ho').text, 'Ho ho')

test_eq(cli.get('/products/products?id=7').text, 'product 7')
test_eq(Products.to(id=7), '/products/products?id=7')

In [ ]:
with cli.websocket_connect('/products/ws') as ws:
    ws.send_text('{"msg":"Hi!"}')
    data = ws.receive_text()
    assert data == 'Message text was: Hi!'

In [ ]:
#| export
for o in all_meths: setattr(APIRouter, o, partialmethod(APIRouter.__call__, methods=o))

In [ ]:
@ar.get
def hi2(): return 'Hi there'
@ar.get("/hi3")
def _(): return 'Hi there'
@ar.post("/post2")
def _(): return 'Postal'

@ar2.get
def hi2(): return 'Hi there'
@ar2.get("/hi3")
def _(): return 'Hi there'
@ar2.post("/post2")
def _(): return 'Postal'

## Extras

In [ ]:
app,cli,rt = get_cli(FastHTML(secret_key='soopersecret'))

In [ ]:
#| export
def cookie(key: str, value="", max_age=None, expires=None, path="/", domain=None, secure=False, httponly=False, samesite="lax",):
    "Create a 'set-cookie' `HttpHeader`"
    cookie = cookies.SimpleCookie()
    cookie[key] = value
    if max_age is not None: cookie[key]["max-age"] = max_age
    if expires is not None: cookie[key]["expires"] = format_datetime(expires, usegmt=True) if isinstance(expires, datetime) else expires
    if path is not None: cookie[key]["path"] = path
    if domain is not None: cookie[key]["domain"] = domain
    if secure: cookie[key]["secure"] = True
    if httponly: cookie[key]["httponly"] = True
    if samesite is not None:
        assert samesite.lower() in [ "strict", "lax", "none", ], "must be 'strict', 'lax' or 'none'"
        cookie[key]["samesite"] = samesite
    cookie_val = cookie.output(header="").strip()
    return HttpHeader("set-cookie", cookie_val)

In [ ]:
@rt("/setcookie")
def get(req): return cookie('now', datetime.now())

@rt("/getcookie")
def get(now:parsed_date): return f'Cookie was set at time {now.time()}'

print(cli.get('/setcookie').text)
time.sleep(0.01)
cli.get('/getcookie').text

In [ ]:
#| export
def reg_re_param(m, s):
    "Register a Starlette URL convertor named `m` matching regex `s`"
    cls = get_class(f'{m}Conv', sup=StringConvertor, regex=s)
    register_url_convertor(m, cls())

In [ ]:
#| export
# Starlette doesn't have the '?', so it chomps the whole remaining URL
reg_re_param("path", ".*?")
_static_exts = "ico gif jpg jpeg webm css js woff png svg mp4 webp ttf otf eot woff2 txt html map pdf zip tgz gz csv mp3 wav ogg flac aac doc docx xls xlsx ppt pptx epub mobi bmp tiff avi mov wmv mkv xml yaml yml rar 7z tar bz2 htm xhtml apk dmg exe msi swf iso".split()
reg_re_param("static", '|'.join(_static_exts))

@patch
def static_route_exts(self:FastHTML, prefix='/', static_path='.', exts='static'):
    "Add a static route at URL path `prefix` with files from `static_path` and `exts` defined by `reg_re_param()`"
    @self.get(f"{prefix}{{fname:path}}.{{ext:{exts}}}")
    async def get(fname:str, ext:str): return FileResponse(f'{static_path}/{fname}.{ext}')

In [ ]:
reg_re_param("imgext", "ico|gif|jpg|jpeg|webm|pdf")

@rt(r'/static/{path:path}{fn}.{ext:imgext}')
def get(fn:str, path:str, ext:str): return f"Getting {fn}.{ext} from /{path}"

test_r(cli, '/static/foo/jph.me.ico', 'Getting jph.me.ico from /foo/')

In [ ]:
app.static_route_exts()
assert 'These are the source notebooks for FastHTML' in cli.get('/README.txt').text

In [ ]:
#| export
@patch
def static_route(self:FastHTML, ext='', prefix='/', static_path='.'):
    "Add a static route at URL path `prefix` with files from `static_path` and single `ext` (including the '.')"
    @self.get(f"{prefix}{{fname:path}}{ext}")
    async def get(fname:str): return FileResponse(f'{static_path}/{fname}{ext}')

In [ ]:
app.static_route('.md', static_path='../..')
assert 'THIS FILE WAS AUTOGENERATED' in cli.get('/README.md').text

In [ ]:
#| export
class StaticNoCache(StaticFiles):
    "StaticFiles that sends `Cache-Control: no-cache`, so browsers revalidate on every use"
    def file_response(self, *args, **kwargs):
        resp = super().file_response(*args, **kwargs)
        resp.headers.setdefault("Cache-Control", "no-cache")
        return resp

In [ ]:
#| export
class StaticImmutable(StaticFiles):
    "Like `StaticNoCache`, but marks responses as cacheable forever: for versioned URLs such as `vurl` links or content-hashed filenames"
    def file_response(self, *args, **kwargs):
        resp = super().file_response(*args, **kwargs)
        resp.headers.setdefault("Cache-Control", "public, max-age=31536000, immutable")
        return resp

These two classes are the poles of static-file caching. `StaticNoCache` makes the browser revalidate on every use: always fresh at the cost of a conditional request each time, right for files that change under a stable URL (and for dev, where you want edits visible on reload). `StaticImmutable` promises the URL's content will never change, so the browser may cache it forever with no requests at all: only safe when a changed file gets a changed URL, which is what `vurl` or a bundler's content-hashed filenames provide.

In [ ]:
statd = Path(tempfile.mkdtemp())
(statd/'app.css').write_text('body{}')
stat_app = FastHTML(routes=[Mount('/nc', StaticNoCache(directory=statd)), Mount('/imm', StaticImmutable(directory=statd))])
stat_cli = TestClient(stat_app)
test_eq(stat_cli.get('/nc/app.css').headers['cache-control'], 'no-cache')
test_eq(stat_cli.get('/imm/app.css').headers['cache-control'], 'public, max-age=31536000, immutable')

In [ ]:
#| export
@flexicache(mtime_policy(arg=0))
def _vhash(p): return hashlib.blake2b(p.read_bytes(), digest_size=4).hexdigest()

def vurl(path, root='.'):
    "Return `path` as a URL with a `v` query param hashing the file's contents, so the URL changes when the file does"
    u = urlsplit(str(path))
    p = Path(root)/u.path.lstrip('/')
    q = f'{u.query}&' if u.query else ''
    return urlunsplit(u._replace(query=f'{q}v={_vhash(p)}'))


Browsers use the URL as the cache key, so a URL that changes whenever the file's contents change can be cached forever: serve the mount with `StaticImmutable` and the asset costs no requests at all until it is next edited. This is the query-param version of the content-hashed filenames that asset pipelines and bundlers produce, without the build step. The hash is cached via `flexicache` with `mtime_policy`, so a call costs one `stat` and the file is only re-read after it changes. The param is inert under `no-cache` serving, so the same links work in dev with `StaticNoCache`, where the file is rechecked on each reload.

In [ ]:
tmp = Path(tempfile.mkdtemp())
(tmp/'a.css').write_text('body{}')
u = vurl('a.css', tmp)
assert u.startswith('a.css?v=')
test_eq(u, vurl('a.css', tmp))
test_eq(vurl('/a.css', tmp).split('?')[0], '/a.css')
q = vurl('/a.css?theme=dark#top', tmp)
assert q.startswith('/a.css?theme=dark&v=') and q.endswith('#top')
(tmp/'a.css').write_text('body{color:red}')
os.utime(tmp/'a.css', (time.time()+1,)*2)
assert vurl('a.css', tmp) != u

In [ ]:
#| export
from functools import wraps
from inspect import signature, isawaitable

In [ ]:
#| export
def add_sig_param(f, name, typ=NoneType, kind=Parameter.KEYWORD_ONLY, default=Parameter.empty):
    "Add a parameter to a function's signature"
    sig = signature(f)
    if name in sig.parameters: return f
    kw = {} if default is Parameter.empty else {'default': default}
    new_params = list(sig.parameters.values()) + [Parameter(name, kind, annotation=typ, **kw)]
    f.__signature__ = sig.replace(parameters=new_params)
    f.__annotations__[name] = typ
    return f

In [ ]:
#| export
class into:
    "Decorator to pass a route's return value into `func`, with keyword params added to the route signature"
    def __init__(self, func):
        self.func = func
        self.params = {k:p for k,p in signature(func).parameters.items()
            if p.kind not in (Parameter.VAR_POSITIONAL, Parameter.VAR_KEYWORD)}

    def __call__(self, f):
        @wraps(f)
        async def _inner(*args, **kw):
            extra = {k: kw.pop(k, None) for k in self.params}
            res = f(*args, **kw)
            if isawaitable(res): res = await res
            res = self.func(*tuplify(res), **extra)
            if isawaitable(res): res = await res
            return res
        for k,p in self.params.items():
            anno = p.annotation if p.annotation is not Parameter.empty else NoneType
            add_sig_param(_inner, k, anno, default=p.default)
        return _inner

`into` is a decorator that composes two functions: the decorated route function and an outer "wrapper" function (like a layout). The route's return value is passed as positional args to the wrapper. Any keyword parameters listed in `params` are added to the route's signature, so they become available as query/form parameters. This lets you write simple routes that return just their content, while the layout and its parameters are handled automatically.

Here `hello` returns a single `P` tag, but `into(my_layout)` passes it into `my_layout`, which wraps it in a `Titled`. The `title` parameter is injected into `hello`'s signature so it can be set via query params.

In [ ]:
app,cli,rt = get_cli(FastHTML())

def my_layout(*c, title:str='Default'): return Titled(title, *c)

@rt
@into(my_layout)
def hello(name:str='World'): return P(f'Hello, {name}!')

With no query params, `name` defaults to `'World'` and `title` defaults to `'Default'` (from `my_layout`'s own default), giving us a page titled "Default" containing "Hello, World!".

In [ ]:
r = cli.get('/hello')
assert '<title>Default</title>' in r.text
assert '<p>Hello, World!</p>' in r.text

Here, both `name` and `title` are provided as query params. `name='Jeremy'` is handled by `hello` directly, while `title='Greetings'` is the injected param that gets forwarded to `my_layout`.

In [ ]:
r = cli.get('/hello?name=Jeremy&title=Greetings')
assert '<title>Greetings</title>' in r.text
assert '<p>Hello, Jeremy!</p>' in r.text

In [ ]:
#| export
class MiddlewareBase:
    "Base for middleware: passes non-http/ws scopes through, else returns the `HTTPConnection`"
    async def __call__(self, scope, receive, send) -> None:
        if scope["type"] in ("http", "websocket"): return HTTPConnection(scope)
        await self._app(scope, receive, send)
        

In [ ]:
#| export
class FtResponse:
    "Wrap an FT response with any Starlette `Response`"
    def __init__(self, content, status_code:int=200, headers=None, cls=HTMLResponse, media_type:str|None=None, background: BackgroundTask | None = None):
        self.content,self.status_code,self.headers = content,status_code,headers
        self.cls,self.media_type,self.background = cls,media_type,background

    def __response__(self, req):
        resp,kw = _part_resp(req, self.content)
        cts = _xt_cts(req, resp)
        tasks,httphdrs = kw.get('background'),kw.get('headers')
        if not tasks: tasks = self.background
        headers = {**(self.headers or {}), **httphdrs}
        return self.cls(cts, status_code=self.status_code, headers=headers, media_type=self.media_type, background=tasks)

In [ ]:
@rt('/ftr')
def get():
    cts = Title('Foo'),H1('bar')
    return FtResponse(cts, status_code=201, headers={'Location':'/foo/1'})

r = cli.get('/ftr')

test_eq(r.status_code, 201)
test_eq(r.headers['location'], '/foo/1')
txt = r.text
assert '<title>Foo</title>' in txt and '<h1>bar</h1>' in txt and '<html>' in txt

Test on a single background task:

In [ ]:
def my_slow_task():
    print('Starting slow task')    
    time.sleep(0.001)
    print('Finished slow task')        

@rt('/background')
def get(): return P('BG Task'), BackgroundTask(my_slow_task)

r = cli.get('/background')

test_eq(r.status_code, 200)



Test multiple background tasks:

In [ ]:
def increment(amount):
    amount = amount/1000
    print(f'Sleeping for {amount}s')    
    time.sleep(amount)
    print(f'Slept for {amount}s')     

In [ ]:
@rt
def backgrounds():
    tasks = BackgroundTasks()
    for i in range(3): tasks.add_task(increment, i)
    return P('BG Tasks'), tasks

r = cli.get('/backgrounds')
test_eq(r.status_code, 200)

In [ ]:
@rt
def backgrounds2():
    tasks = [BackgroundTask(increment,i) for i in range(3)]
    return P('BG Tasks'), *tasks

r = cli.get('/backgrounds2')
test_eq(r.status_code, 200)

In [ ]:
@rt
def backgrounds3():
    tasks = [BackgroundTask(increment,i) for i in range(3)]
    return {'status':'done'}, *tasks

r = cli.get('/backgrounds3')
test_eq(r.status_code, 200)
r.json()

In [ ]:
#| export
def unqid(seeded=False):
    "Random unique id, base64-encoded and prefixed with '_' so it's usable as an HTML id"
    id4 = UUID(int=random.getrandbits(128), version=4) if seeded else uuid4()
    res = b64encode(id4.bytes)
    return '_' + res.decode().rstrip('=').translate(str.maketrans('+/', '_-'))

In [ ]:
#| export
def _add_ids(s):
    if not isinstance(s, FT): return
    if not getattr(s, 'id', None): s.id = unqid()
    for c in s.children: _add_ids(c)

In [ ]:
#| export
@patch
def setup_ws(app:FastHTML, f=noop):
    conns = {}
    async def on_connect(ws, send): conns[ws.client] = send
    async def on_disconnect(ws): conns.pop(ws.client)
    app.ws('/ws', conn=on_connect, disconn=on_disconnect)(f)
    async def send(s):
        for o in conns.values(): await o(s)
    app._send = send
    return send

`setup_ws` returns an async function that broadcasts FT elements to connected clients. Here a message handler broadcasts each incoming message. Clients can disconnect and reconnect:

In [ ]:
push_app = FastHTML()
async def relay(msg:str): await push(Div(msg, id='status'))
push = push_app.setup_ws(relay)
with TestClient(push_app) as client:
    for msg in ('hello', 'again'):
        with client.websocket_connect('/ws') as ws:
            ws.send_json(dict(msg=msg))
            assert msg in ws.receive_text()

In [ ]:
#| export
devtools_loc = "/.well-known/appspecific/com.chrome.devtools.json"

@patch
def devtools_json(self:FastHTML, path=None, uuid=None):
    if not path: path = Path().absolute()
    if not uuid: uuid = str(uuid5(NAMESPACE_URL, str(path)))
    @self.route(devtools_loc)
    def devtools(): return dict(workspace=dict(root=str(path), uuid=uuid))

In [ ]:
#| export
@patch
def get_client(self:FastHTML, asink=False, **kw):
    "Get an httpx client with session cookes set from `**kw`"
    import httpx2
    signer = itsdangerous.TimestampSigner(self.secret_key)
    data = b64encode(dumps(kw).encode())
    data = signer.sign(data)
    client = httpx2.AsyncClient() if asink else httpx2.Client()
    client.cookies.update({self.session_cookie: data.decode()})
    return client

In [ ]:
#| export
@patch
def decode_session(self:FastHTML, cookie):
    "Decode a signed session cookie"
    if not cookie or cookie == 'null': return {}
    unsigned = itsdangerous.TimestampSigner(self.secret_key).unsign(cookie, max_age=None)
    return loads(b64decode(unsigned).decode())

In [ ]:
#| export
@patch
def get_testclient(self:FastHTML, **kw):
    "Get a Starlette `TestClient` with session cookies set from `**kw`"
    from starlette.testclient import TestClient
    
    class FastHTMLTestClient(TestClient):
        "A Starlette TestClient with a `session` property"
        @property
        def session(self):
            cookie = next((c.value for c in reversed(list(self.cookies.jar)) if c.name == self.app.session_cookie), None)
            return self.app.decode_session(cookie)

    return FastHTMLTestClient(self, cookies=self.get_client(**kw).cookies)

In [ ]:
cli = app.get_testclient(auth='user1')
test_eq(cli.session['auth'], 'user1')

## Export -

In [ ]:
#| hide
import nbdev
nbdev.nbdev_export()